# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: 
* Model: 
* Evaluation approach: 
* Fine-tuning dataset: 

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
#PEFT technique: LoRA
#Model: GPT-2 from Hugging Face, adapted for sequence classification
#Evaluation approach: Use the Hugging Face Trainer with the accuracy metric to evaluate classification performance.
#Fine-tuning dataset: HaluEval dataset from Hugging Face. The dataset has examples of hallucinated
#and real responses generated by language model. This was chosen for this project because it gives me the information needed to train a model
#to classify whether a response is hallucinated or not.

In [2]:
# Load tokenizer and libraries
!pip install transformers datasets peft evaluate scikit-learn -q
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, AutoPeftModelForSequenceClassification
import evaluate

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
# Load dataset
from datasets import load_dataset, Dataset
dataset = load_dataset("pminervini/HaluEval", "dialogue")
full_data = dataset["data"]

# Train and test the dataset
train_data = full_data.select(range(2000))
test_data = full_data.select(range(2000, 2500))

# Binary classification (0 = real, 1 = hallucinated)
def create_binary_dataset(data):
    texts, labels = [], []
    for ex in data:
        texts.append(ex["dialogue_history"] + " " + ex["right_response"])
        labels.append(0)
        texts.append(ex["dialogue_history"] + " " + ex["hallucinated_response"])
        labels.append(1)
    return Dataset.from_dict({"text": texts, "label": labels})

train_dataset = create_binary_dataset(train_data)
test_dataset = create_binary_dataset(test_data)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Train size: 4000
Test size: 1000


In [4]:
# Tokenize gpt2 dataset
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Truncate the tokenized dataset and set format to pyTorch
def tokenize(example):
    return tokenizer(example["text"], truncation = True, padding="max_length", max_length = 128)

train_dataset = train_dataset.map(tokenize, batched = True)
test_dataset = test_dataset.map(tokenize, batched = True)

train_dataset.set_format("torch")
test_dataset.set_format("torch")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
# Load gpt2 for sequence classification
model = AutoModelForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

# Compute metrics during eval
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    return accuracy.compute(predictions = predictions, references = labels)

# Initialize trainer with model, eval dataset, and metric
args = TrainingArguments(output_dir="results", per_device_eval_batch_size = 8)
trainer = Trainer( model = model, args = args, eval_dataset = test_dataset, compute_metrics = compute_metrics)

original_results = trainer.evaluate()
print("Original:", original_results)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Original: {'eval_loss': 0.7646588087081909, 'eval_accuracy': 0.506, 'eval_runtime': 7.8622, 'eval_samples_per_second': 127.192, 'eval_steps_per_second': 15.899}


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [6]:
# Lora configuration for parameter-efficient fine-tuning
config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["c_attn"],
    lora_dropout = 0.1,
    task_type = "SEQ_CLS"
)

In [7]:
# Convert pretrained gpt2 model into a peft model and print trainable vs total parameters
peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters()

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 297,984 || all params: 124,737,792 || trainable%: 0.23888830740245906


In [8]:
# Training arguments for peft model
args = TrainingArguments(
    output_dir = "results",
    learning_rate = 2e-5,
    per_device_train_batch_size = 4,
    num_train_epochs = 1,
    evaluation_strategy = "epoch",
    save_strategy = "epoch"
)

# Initialize hugging face trainer
trainer = Trainer(
    model = peft_model,
    args = args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.670100,0.664225,0.621000


Checkpoint destination directory results/checkpoint-1000 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=1000, training_loss=0.6851063842773437, metrics={'train_runtime': 89.091, 'train_samples_per_second': 44.898, 'train_steps_per_second': 11.224, 'total_flos': 262207438848000.0, 'train_loss': 0.6851063842773437, 'epoch': 1.0})

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [10]:
# Saving the model
peft_model.save_pretrained("/tmp/gpt2-lora-hallucination")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [12]:
# Load the new fine tuned peft (loRa) and set to eval
lora_model = AutoPeftModelForSequenceClassification.from_pretrained("gpt2-lora-hallucination")
lora_model.eval()

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): GPT2ForSequenceClassification(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): Linear(
                in_features=768, out_features=2304, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_

In [13]:
# Evaluate Lora using prev evaluation setup
trainer = Trainer(
    model = model,
    args = args,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)

fine_tuned_results = trainer.evaluate()
print("Fine-tuned:", fine_tuned_results)

Fine-tuned: {'eval_loss': 0.6642246246337891, 'eval_accuracy': 0.621, 'eval_runtime': 8.5519, 'eval_samples_per_second': 116.933, 'eval_steps_per_second': 14.617}


In [14]:
# Print side by side comparison of pre trained model and fine tuned model
print("Original Model:", original_results)
print("Fine-Tuned Model:", fine_tuned_results)

Original Model: {'eval_loss': 0.7646588087081909, 'eval_accuracy': 0.506, 'eval_runtime': 7.8622, 'eval_samples_per_second': 127.192, 'eval_steps_per_second': 15.899}
Fine-Tuned Model: {'eval_loss': 0.6642246246337891, 'eval_accuracy': 0.621, 'eval_runtime': 8.5519, 'eval_samples_per_second': 116.933, 'eval_steps_per_second': 14.617}
